# 01 Tensor 和形状

本 notebook 对应学习计划的第 1 阶段。目标不是记 API，而是能从第一性原理理解 Tensor 的形状、内存布局、设备和常见变换。

## 本节目标

完成后应该能够：

- 看到一个 Tensor 后说清楚 `shape`、`dtype`、`device`、`stride` 和是否连续。
- 手动推导 `reshape`、`view`、`permute`、`transpose` 后的形状。
- 判断广播是否成立。
- 写出 batch 矩阵乘法的形状关系。
- 解释 NumPy 和 Tensor 转换时的共享内存与拷贝。
- 解释为什么不连续 Tensor 直接 `.view()` 可能失败。

In [6]:
from pathlib import Path
import sys

import numpy as np
import torch

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from learn_pytorch import get_device

torch.manual_seed(42)

device = get_device()
print(f"torch version: {torch.__version__}")
print(f"selected device: {device}")

torch version: 2.12.0
selected device: mps


In [10]:
def describe(name: str, tensor: torch.Tensor) -> None:
    """Print the properties that matter most when debugging Tensor shape issues."""
    print(
        f"{name}: "
        f"shape={tuple(tensor.shape)}, "
        f"ndim={tensor.ndim}, "
        f"dtype={tensor.dtype}, "
        f"device={tensor.device}, "
        f"stride={tensor.stride()}, "
        f"contiguous={tensor.is_contiguous()}"
    )

## 1. Tensor 的基本属性

调试 Tensor 时优先看五件事：`shape`、`ndim`、`dtype`、`device`、`stride`。其中 `stride` 反映底层内存布局，后面理解 `view()` 会用到。

In [11]:
scalar = torch.tensor(3.14)
vector = torch.arange(5)
matrix = torch.arange(12).reshape(3, 4)
batch_images = torch.zeros(2, 3, 4, 5)

describe("scalar", scalar)
describe("vector", vector)
describe("matrix", matrix)
describe("batch_images", batch_images)

scalar: shape=(), ndim=0, dtype=torch.float32, device=cpu, stride=(), contiguous=True
vector: shape=(5,), ndim=1, dtype=torch.int64, device=cpu, stride=(1,), contiguous=True
matrix: shape=(3, 4), ndim=2, dtype=torch.int64, device=cpu, stride=(4, 1), contiguous=True
batch_images: shape=(2, 3, 4, 5), ndim=4, dtype=torch.float32, device=cpu, stride=(60, 20, 5, 1), contiguous=True


常见形状约定：

```text
标量：[]
向量：[D]
矩阵：[N, D]
图像 batch：[B, C, H, W]
序列 batch：[B, S] 或 [B, S, E]
分类 logits：[B, num_classes]
```

## 2. 形状变换：reshape、view、permute、transpose

第一性原理：`shape` 是逻辑视图，`stride` 是底层内存步长。只改逻辑形状不一定需要复制数据，但维度交换常常会让 Tensor 变成不连续。

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)

describe("x", x)
describe("x.reshape(6, 4)", x.reshape(6, 4))
describe("x.view(2, 12)", x.view(2, 12))
describe("x.permute(0, 2, 1)", x.permute(0, 2, 1))
describe("x.transpose(1, 2)", x.transpose(1, 2))

`view()` 要求底层内存布局兼容。`transpose()` 或 `permute()` 后的 Tensor 往往不连续，直接 `view()` 可能失败。

In [ ]:
y = x.transpose(1, 2)
describe("y = x.transpose(1, 2)", y)

try:
    y.view(2, 12)
except RuntimeError as error:
    print("view failed:", error)

describe("y.contiguous().view(2, 12)", y.contiguous().view(2, 12))
describe("y.reshape(2, 12)", y.reshape(2, 12))

## 3. 广播

广播从最后一个维度开始对齐。两个维度要么相等，要么其中一个是 `1`，要么其中一个 Tensor 没有这个维度。

In [ ]:
scores = torch.arange(6, dtype=torch.float32).reshape(2, 3)
bias = torch.tensor([10.0, 20.0, 30.0])
result = scores + bias

describe("scores", scores)
describe("bias", bias)
describe("scores + bias", result)
print(result)

assert result.shape == scores.shape

In [ ]:
column_bias = torch.tensor([[100.0], [200.0]])
broadcasted = scores + column_bias

describe("column_bias", column_bias)
describe("scores + column_bias", broadcasted)
print(broadcasted)

assert broadcasted.shape == scores.shape

## 4. Batch 矩阵乘法

二维矩阵乘法：`[N, M] @ [M, P] -> [N, P]`。

Batch 矩阵乘法：`[B, N, M] @ [B, M, P] -> [B, N, P]`。最后两个维度做矩阵乘法，前面的维度当作 batch。

In [ ]:
B, N, M, P = 2, 3, 4, 5
a = torch.randn(B, N, M)
b = torch.randn(B, M, P)
c = torch.bmm(a, b)

describe("a", a)
describe("b", b)
describe("torch.bmm(a, b)", c)

torch.testing.assert_close(c[0], a[0] @ b[0])
torch.testing.assert_close(c[1], a[1] @ b[1])
print("batch matmul result is consistent with per-sample matmul")

## 5. 设备选择

本项目把设备选择沉淀到了 `src/learn_pytorch/device.py`。默认优先级是 `cuda -> mps -> cpu`。学习阶段不要把硬件写死。

In [12]:
device = get_device()
x_cpu = torch.arange(4, dtype=torch.float32)
x_device = x_cpu.to(device)

describe("x_cpu", x_cpu)
describe("x_device", x_device)

print("explicit cpu:", get_device("cpu"))
print("requested cuda fallback:", get_device("cuda"))
print("requested mps fallback:", get_device("mps"))

x_cpu: shape=(4,), ndim=1, dtype=torch.float32, device=cpu, stride=(1,), contiguous=True
x_device: shape=(4,), ndim=1, dtype=torch.float32, device=mps:0, stride=(1,), contiguous=True
explicit cpu: cpu
requested cuda fallback: cpu
requested mps fallback: mps


注意：`tensor.to(device)` 返回的是移动后的 Tensor，不要假设原变量一定被原地修改。模型参数和输入 Tensor 必须在同一设备。

## 6. NumPy 与 Tensor 的内存共享

`torch.from_numpy()` 和 `torch.as_tensor()` 通常会共享 NumPy 数组的底层内存；`torch.tensor()` 和 `.clone()` 会复制数据。

In [ ]:
array = np.arange(6, dtype=np.float32).reshape(2, 3)

from_numpy = torch.from_numpy(array)
as_tensor = torch.as_tensor(array)
copied = torch.tensor(array)
cloned = from_numpy.clone()

array[0, 0] = 100.0
from_numpy[0, 1] = 200.0

print("numpy array:\n", array)
print("from_numpy shares memory:\n", from_numpy)
print("as_tensor shares when possible:\n", as_tensor)
print("torch.tensor copied before mutation:\n", copied)
print("clone copied before mutation:\n", cloned)

如果后续会修改 NumPy array，必须明确自己想共享还是复制。数据预处理阶段很多隐蔽 bug 都来自这里。

## 7. 本节复盘

把后续训练问题拆回第一性原理，通常就是：

```text
这个 Tensor 是什么形状？
它在哪里？
它是什么 dtype？
它和另一个 Tensor 是否能广播或矩阵相乘？
它的内存布局是否支持当前 view？
它是否和外部 NumPy array 共享底层内存？
```

In [ ]:
# Quick self-check: these assertions summarize the core shape rules in this notebook.
assert torch.zeros(2, 3, 4).permute(0, 2, 1).shape == torch.Size([2, 4, 3])
assert (torch.zeros(2, 3) + torch.zeros(3)).shape == torch.Size([2, 3])
assert (torch.zeros(2, 3, 4) @ torch.zeros(2, 4, 5)).shape == torch.Size([2, 3, 5])
assert get_device("cpu") == torch.device("cpu")
print("stage 1 self-check passed")